In [1]:
# Importing libraries

import requests
import os
import pandas as pd

from census import Census
from us import states
from dotenv import load_dotenv

In [6]:
# Some os work with a .env file to ensure security of API Key
load_dotenv()

API_KEY = os.getenv("CENSUS_API_KEY")

In [ ]:
# Creating dataframes
state_and_county_df = pd.read_csv("../data/raw/StateAndCountyData.csv")


,FIPS,Value
count,957278.000000,9.577530e+05
mean,30343.740874,1.114661e+04
std,15183.939474,6.302679e+05
min,1001.000000,-9.999000e+03
25%,18169.000000,0.000000e+00
50%,29171.000000,2.738841e+00
75%,45079.000000,3.058333e+01
max,56045.000000,3.380466e+08


In [7]:
# Top 5 rows
print(state_and_county_df.head())

# Descriptive values
print(state_and_county_df.describe())

# Other valuable information
print(state_and_county_df.info())

     FIPS State   County          Variable_Code        Value
0  1001.0    AL  Autauga          LACCESS_POP15  18092.66135
1  1001.0    AL  Autauga          LACCESS_POP19  18503.22551
2  1001.0    AL  Autauga  PCH_LACCESS_POP_15_19      2.26923
3  1001.0    AL  Autauga      PCT_LACCESS_POP15     33.15435
4  1001.0    AL  Autauga      PCT_LACCESS_POP19     33.90670
                FIPS         Value
count  957278.000000  9.577530e+05
mean    30343.740874  1.114661e+04
std     15183.939474  6.302679e+05
min      1001.000000 -9.999000e+03
25%     18169.000000  0.000000e+00
50%     29171.000000  2.738841e+00
75%     45079.000000  3.058333e+01
max     56045.000000  3.380466e+08
<class 'pandas.DataFrame'>
RangeIndex: 957753 entries, 0 to 957752
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   FIPS           957278 non-null  float64
 1   State          957753 non-null  str    
 2   County         957753 non-null  str

In [ ]:
# Importing census data
c = Census(API_KEY)

# Pulling the Washington State Data
wa_data = c.acs5.state_county_tract(
    fields = (
        "NAME", 
        "B19013_001E",   # Median household income
        "B25044_003E",   # Owner units, no vehicle
        "B17001_002E",   # Population below poverty level
        "B17001_001E",   # Total population 
        "B25044_010E",   # Renter units, no vehicle
        "B25044_001E",   # Total occupied units
        "B03002_003E",   # Non-Hispanic white
        "B03002_004E",   # Black/African American
        "B03002_012E",   # Hispanic/Latino
        "B03002_001E",   # Total population
        "B22003_002E",   # SNAP households
        "B22003_001E",   # Total households
    ),
    state_fips = states.WA.fips, 
    county_fips = Census.ALL, 
    tract = Census.ALL
)

wa_vehicle_df = pd.DataFrame(wa_data)

In [ ]:
# Get the poverty rate by dividing the poverty count by the total population
wa_vehicle_df["poverty_rate"] = wa_vehicle_df["B17001_002E"] / wa_vehicle_df["B17001_001E"]

# Getting GEOID - Which is the geographical identifier listed by the Census Bureau
# This is an 11 number combination between the state, county, and tract
wa_vehicle_df["GEOID"] = wa_vehicle_df["state"] + wa_vehicle_df["county"] + wa_vehicle_df["tract"]

# 

In [19]:
print(wa_vehicle_df.head())

                                             NAME  B19013_001E  B25044_003E  \
0     Census Tract 9501; Adams County; Washington      68750.0          6.0   
1     Census Tract 9502; Adams County; Washington      72500.0         14.0   
2  Census Tract 9503.01; Adams County; Washington      58585.0          0.0   
3  Census Tract 9503.02; Adams County; Washington      65893.0         15.0   
4  Census Tract 9503.03; Adams County; Washington      75673.0          0.0   

   B17001_002E  B17001_001E  B25044_010E  B25044_001E  B03002_003E  \
0        314.0       2681.0         57.0       1111.0       2281.0   
1        267.0       2041.0          0.0        610.0       1199.0   
2        117.0       1762.0          0.0        593.0        576.0   
3        913.0       2981.0         35.0        779.0        264.0   
4        315.0       2384.0          0.0        697.0        578.0   

   B03002_004E  B03002_012E  B03002_001E  B22003_002E  B22003_001E state  \
0         16.0        144.0 